# Logistic regression

_Machine Learning_

**Squash a score into a probability between 0 and 1. Classify by it.**

For yes/no questions, we want a probability, not just any number.

     Linear regression can output anything, even $-50$ or $1000$. That is no good for a probability.

---

This notebook is a practice scaffold for the **Logistic regression** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

### Step 1 — Define the model, loss, and optimizer

A logistic-regression model is just a single linear layer that maps 2 features to 1 score (a *logit*). We pair it with `BCEWithLogitsLoss`, which applies the sigmoid and binary cross-entropy together in a numerically stable way, and an SGD optimizer that will nudge the weights downhill. We never apply the sigmoid by hand during training — the loss does it for us.

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(2, 1)              # weights w and bias b
loss_fn = nn.BCEWithLogitsLoss()    # sigmoid + cross-entropy, numerically stable
opt = torch.optim.SGD(model.parameters(), lr=0.1)

### Step 2 — Generate a linearly separable dataset

We draw 200 random points and label each one by whether `x0 + x1 > 0`. That gives a clean diagonal decision boundary the model should be able to recover. We reshape the labels to a column vector with `unsqueeze(1)` so their shape matches the model's output.

In [ ]:
X = torch.randn(200, 2)
y = (X[:, 0] + X[:, 1] > 0).float().unsqueeze(1)   # true boundary

### Step 3 — Train with the gradient-descent loop

Each epoch we run the standard PyTorch loop: zero the old gradients, compute the logits and loss, call `backward()` to let autograd fill in the gradients, then `step()` to update the weights. Over 100 epochs the weights converge toward the true boundary.

In [ ]:
for epoch in range(100):
    opt.zero_grad()
    logits = model(X)
    loss = loss_fn(logits, y)
    loss.backward()                 # autograd computes gradients
    opt.step()                      # gradient-descent update

### Step 4 — Measure accuracy

To turn logits into predictions we apply the sigmoid and threshold at 0.5: scores above 0.5 become class 1, the rest class 0. Comparing those predictions to the true labels and averaging gives the training accuracy, which should be close to 1.0 on this cleanly separable data.

In [ ]:
preds = (torch.sigmoid(model(X)) > 0.5).float()
print("accuracy:", (preds == y).float().mean().item())

## Visualize the data & results

_Using just mean radius and mean texture, where does logistic regression draw the line between malignant and benign?_

### Step 1 — Fit logistic regression on two real features

We use scikit-learn's `LogisticRegression` on two interpretable tumour measurements: mean radius and mean texture. After fitting, the training accuracy tells us how well these two features alone separate malignant from benign tumours.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

# two real, interpretable features of each tumor
bc = load_breast_cancer()
X = bc.data[:, [0, 1]]             # mean radius, mean texture
y = bc.target
clf = LogisticRegression(max_iter=5000).fit(X, y)
print("accuracy", clf.score(X, y))

### Step 2 — Derive the decision boundary line

The classifier predicts class 1 wherever `w0·x + w1·z + b = 0`. Solving that equation for `z` gives `z = -(w0·x + b) / w1`, a straight line in feature space. We read the learned weights and intercept off the fitted model, then evaluate this line across the range of mean-radius values.

In [ ]:
# decision boundary: w0*x + w1*z + b = 0  ->  z = -(w0*x + b)/w1
w0, w1 = clf.coef_[0]
b = clf.intercept_[0]
xs = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
zs = -(w0 * xs + b) / w1

### Step 3 — Plot the classes and the boundary

We scatter the malignant and benign tumours in different colours and overlay the decision boundary line. Points fall mostly on the correct side, but some overlap near the line — exactly the cases where the classifier is uncertain and accuracy drops below 100%.

In [ ]:
for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = X[y == label]
    plt.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")
plt.plot(xs, zs, color="#ffb454", label="boundary")
plt.xlabel("mean radius")
plt.ylabel("mean texture")
plt.title("Logistic regression boundary")
plt.legend()
plt.show()